Step 1 院校数据Data Description

In [1]:
import pandas as pd

path = "../data/raw/Most-Recent-Cohorts-Institution.csv"
inst = pd.read_csv(path, dtype=str, keep_default_na=False)

print(inst.shape)

(6273, 3308)


In [2]:
inst.iloc[:5, :12]

,UNITID,OPEID,OPEID6,INSTNM,CITY,STABBR,ZIP,ACCREDAGENCY,INSTURL,NPCURL,SCH_DEG,HCM2
0,100654,00100200,001002,Alabama A & M University,Normal,AL,35762,Southern Association of Colleges and Schools C...,www.aamu.edu/,www.aamu.edu/admissions-aid/tuition-fees/net-p...,3,0
1,100663,00105200,001052,University of Alabama at Birmingham,Birmingham,AL,35294-0110,Southern Association of Colleges and Schools C...,https://www.uab.edu/,https://tcc.ruffalonl.com/University of Alabam...,3,0
2,100690,02503400,025034,Amridge University,Montgomery,AL,36117-3553,Southern Association of Colleges and Schools C...,https://www.amridgeuniversity.edu/,https://www2.amridgeuniversity.edu:9091/,3,0
3,100706,00105500,001055,University of Alabama in Huntsville,Huntsville,AL,35899,Southern Association of Colleges and Schools C...,www.uah.edu/,uah.clearcostcalculator.com/student/default/ne...,3,0
4,100724,00100500,001005,Alabama State University,Montgomery,AL,36104-0271,Southern Association of Colleges and Schools C...,www.alasu.edu/,tcc.ruffalonl.com/Alabama State University/Fre...,3,0


In [ ]:
# Missing values 有哪些写法
values = pd.Series(inst.values.ravel())
values.value_counts().head(15)

# PS- Privacy Suppressed 39% 有数据 但人数太少，为保护隐私不公开
# NA- 没有这项数据，17% 比如社区大学没有SAT数据

PS           8112691
NA           3553703
0            1439233
<=0.20        255595
<=0.10        195994
<=0.05        158581
1             140426
2             121271
0.10-0.19      68058
0.05-0.09      62231
<=0.02         60495
0.10-0.14      59952
0.20-0.39      58437
0.15-0.19      37558
0.20-0.29      35322
Name: count, dtype: int64

In [4]:
print(inst["UNITID"].is_unique)
print(inst["UNITID"].nunique())
print(inst["OPEID6"].nunique())

True
6273
4957


In [5]:
top = inst["OPEID6"].value_counts().head(5)
print(top)


OPEID6
022606    31
001459    27
030837    23
003329    22
NA        21
Name: count, dtype: int64


In [8]:
inst[inst["OPEID6"] == "022606"][["UNITID", "INSTNM", "CITY", "STABBR", "CONTROL"]]

,UNITID,INSTNM,CITY,STABBR,CONTROL
3470,242972,NUC University,Bayamon,PR,3
6069,24297201,NUC University - Arecibo,Arecibo,PR,3
6070,24297202,NUC University - Rio Grande,Rio Grande,PR,3
6071,24297203,NUC University - Ponce,Ponce,PR,3
6072,24297204,NUC University - Caguas,Caguas,PR,3
6073,24297205,NUC University - Mayaguez,Mayaguez,PR,3
6074,24297206,NUC University - South Florida,Pembroke Pines,FL,3
6075,24297207,NUC University - Florida Technical College - L...,Lakeland,FL,3
6076,24297208,NUC University - Florida Technical College - D...,Deland,FL,3
6077,24297209,NUC University - Florida Technical College - K...,Kissimmee,FL,3


In [9]:
 inst[inst["OPEID6"] == "001459"][["UNITID", "INSTNM", "CITY", "STABBR", "CONTROL"]]

,UNITID,INSTNM,CITY,STABBR,CONTROL
569,131803,Strayer University-District of Columbia,Washington,DC,3
3246,233684,Strayer University-Virginia,Arlington,VA,3
3988,430184,Strayer University-Maryland,Baltimore,MD,3
4210,443766,Strayer University-Tennessee,Nashville,TN,3
4211,443784,Strayer University-Pennsylvania,Philadelphia,PA,3
4362,449038,Strayer University-Florida,Tampa,FL,3
4406,450298,Strayer University-Delaware,Wilmington,DE,3
4408,450377,Strayer University-Alabama,Birmingham,AL,3
4475,453163,Strayer University-North Carolina,Greensboro,NC,3
4476,453215,Strayer University-New Jersey,Piscataway,NJ,3


### Scope decision (Oct 2026)

Changed the degree filter from PREDDEG in (2, 3) to
PREDDEG in (1, 2, 3) and HIGHDEG in (2, 3, 4).

Reason: PREDDEG only shows the *predominant* award. About 440 public and
nonprofit colleges mainly award certificates but also award associate or
bachelor's degrees, including many community and technical colleges.
325 of them reported registered nursing awards in 2021–23.

Result: 2,557 → 2,991 colleges.
Known limit: 33 colleges with PREDDEG = 1 and HIGHDEG = 4 may not offer
undergraduate degrees; the program lens flags these.

In [19]:
#筛选出目标学校Control=1 公立学校，Control=2 私立非营利学校

steps = []
df = inst.copy()
steps.append(("All institutions", len(df)))

df = df[df["CURROPER"] == "1"]
steps.append(("Currently operating", len(df)))

df = df[df["DISTANCEONLY"] != "1"]
steps.append(("Not online-only", len(df)))

df = df[df["CONTROL"].isin(["1", "2"])]
steps.append(("Public or private nonprofit", len(df)))

df = df[df["PREDDEG"].isin(["1", "2", "3"]) & df["HIGHDEG"].isin(["2", "3", "4"])]
steps.append(("Awards associate's degree or higher, undergrad focus", len(df)))

df = df[df["UGDS"] != "0"]
steps.append(("Has undergraduate students", len(df)))

scope = df
pd.DataFrame(steps, columns=["Step", "Colleges"])

,Step,Colleges
0,All institutions,6273
1,Currently operating,6243
2,Not online-only,6169
3,Public or private nonprofit,3889
4,"Awards associate's degree or higher, undergrad...",2991
5,Has undergraduate students,2989


In [20]:
#检查三个维度的关键字段
academic = ["OPENADMP", "ADM_RATE", "ADMCON7", "SATVR25", "SATVR75",
            "SATMT25", "SATMT75", "ACTCM25", "ACTCM75"]
affordability = ["COSTT4_A", "TUITIONFEE_IN", "TUITIONFEE_OUT",
                 "NPT4_PUB", "NPT4_PRIV", "NPT41_PUB", "NPT41_PRIV"]
preference = ["LOCALE", "UGDS", "LATITUDE", "LONGITUDE"]

MISSING = ["NA", "PS", "NULL", ""]

def check(cols):
    rows = []
    for c in cols:
        v = scope[c]
        missing = v.isin(MISSING)
        as_num = pd.to_numeric(v, errors="coerce")
        not_number = (~missing) & as_num.isna()
        rows.append((c, f"{missing.mean():.1%}", not_number.sum()))
    return pd.DataFrame(rows, columns=["Field", "Missing", "Non-numeric values"])

check(academic + affordability + preference)

#偏好维度最完整：缺失都在 0.3% 以下
#经济维度的公布价格也很完整：学费 3.8%、总成本 6.5%
#学术维度最不完整：SAT 缺失 68.5%

,Field,Missing,Non-numeric values
0,OPENADMP,2.8%,0
1,ADM_RATE,44.5%,0
2,ADMCON7,44.2%,0
3,SATVR25,68.5%,0
4,SATVR75,68.5%,0
5,SATMT25,68.5%,0
6,SATMT75,68.5%,0
7,ACTCM25,69.4%,0
8,ACTCM75,69.4%,0
9,COSTT4_A,6.5%,0


In [21]:
print(scope["OPENADMP"].value_counts())
print(scope["ADMCON7"].value_counts())
pd.crosstab(scope["OPENADMP"], scope["ADM_RATE"].isin(MISSING))

OPENADMP
2     1670
1     1234
NA      85
Name: count, dtype: int64
ADMCON7
NA    1322
5     1024
3      552
1       91
Name: count, dtype: int64


ADM_RATE,False,True
OPENADMP,,
1,0,1234
2,1660,10
NA,0,85


In [22]:
public = scope[scope["CONTROL"] == "1"]
private = scope[scope["CONTROL"] == "2"]

print("Public schools:", len(public), " NPT4_PUB missing:", f"{public['NPT4_PUB'].isin(MISSING).mean():.1%}")
print("Private schools:", len(private), " NPT4_PRIV missing:", f"{private['NPT4_PRIV'].isin(MISSING).mean():.1%}")

Public schools: 1590  NPT4_PUB missing: 2.1%
Private schools: 1399  NPT4_PRIV missing: 10.2%


In [23]:
for c in ["ADM_RATE", "UGDS", "COSTT4_A", "NPT41_PUB", "NPT41_PRIV"]:
    print(c)
    print(pd.to_numeric(scope[c], errors="coerce").describe().round(2))
    print()

ADM_RATE
count    1660.00
mean        0.72
std         0.23
min         0.00
25%         0.60
50%         0.77
75%         0.89
max         1.00
Name: ADM_RATE, dtype: float64

UGDS
count      2979.00
mean       4383.67
std        7598.19
min           2.00
25%         737.00
50%        1807.00
75%        4782.50
max      163164.00
Name: UGDS, dtype: float64

COSTT4_A
count     2796.00
mean     31695.28
std      20006.33
min       4451.00
25%      16357.25
50%      24233.00
75%      43727.25
max      93512.00
Name: COSTT4_A, dtype: float64

NPT41_PUB
count     1557.00
mean      8075.04
std       4301.06
min      -4158.00
25%       4906.00
50%       7818.00
75%      10868.00
max      31963.00
Name: NPT41_PUB, dtype: float64

NPT41_PRIV
count     1216.00
mean     18334.09
std       8696.13
min      -3012.00
25%      12722.25
50%      17881.00
75%      22635.00
max      92434.00
Name: NPT41_PRIV, dtype: float64



In [24]:
scope_n = scope.copy()
for c in ["ADM_RATE", "UGDS", "NPT41_PUB", "NPT41_PRIV"]:
    scope_n[c] = pd.to_numeric(scope_n[c], errors="coerce")

print(scope_n[scope_n["ADM_RATE"] == 0][["INSTNM", "STABBR", "ADM_RATE", "UGDS"]])
print(scope_n[scope_n["UGDS"] == 0][["INSTNM", "STABBR", "HIGHDEG"]])
print(scope_n.nlargest(3, "UGDS")[["INSTNM", "STABBR", "UGDS"]])
print(scope_n[(scope_n["NPT41_PUB"] < 0) | (scope_n["NPT41_PRIV"] < 0)][["INSTNM", "STABBR", "NPT41_PUB", "NPT41_PRIV"]])

                          INSTNM STABBR  ADM_RATE   UGDS
2000  Belanger School of Nursing     NY       0.0  162.0
Empty DataFrame
Columns: [INSTNM, STABBR, HIGHDEG]
Index: []
                                         INSTNM STABBR      UGDS
1796          Southern New Hampshire University     NH  163164.0
80                      Grand Canyon University     AZ   73371.0
68    Arizona State University Campus Immersion     AZ   64674.0
                                     INSTNM STABBR  NPT41_PUB  NPT41_PRIV
185      California Institute of Technology     CA        NaN     -2133.0
218                          Canada College     CA    -1545.0         NaN
345                        Moorpark College     CA    -4158.0         NaN
365                          Oxnard College     CA    -1855.0         NaN
411                    College of San Mateo     CA    -1310.0         NaN
418                 College of the Sequoias     CA     -537.0         NaN
449                         Ventura College    

### Data issues log

| # | Issue | Decision |
|---|---|---|
| 1 | NA and PS mean missing; 56% of all cells | Treat both as missing, never zero |
| 2 | Branch campuses share OPEID6; debt and earnings are repeated across them | Label as system-wide figures if shown |
| 3 | Open admission colleges have no ADM_RATE | Check OPENADMP before ADM_RATE |
| 4 | NPT4_PUB applies to publics only, NPT4_PRIV to privates only | Pick the column by CONTROL |
| 5 | ADM_RATE = 0 (Belanger School of Nursing) | Treat as suspicious: "More info needed" |
| 6 | UGDS = 0 (2 graduate schools) | Excluded from scope |
| 7 | UGDS includes online students (SNHU 163k) | Add an enrollment note; refine later |
| 8 | Negative net price (27 colleges) | Show "$0 or less on average" with explanation |
| 9 | 33 colleges with PREDDEG 1 and HIGHDEG 4 may lack undergrad degrees | Program lens shows "No recent awards" |
| 10 | URLs are inconsistent (missing https, spaces) | Clean before showing links |

In [25]:
scope[["UNITID"]].to_csv("../data/scope_unitids.csv", index=False)